In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

# import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


In [2]:
# import core libraries
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, pipeline
from sentence_transformers import SentenceTransformer, util as st_util
import torch

# Q1. Load train.csv with Hugging Face datasets and create combined_text column

In [3]:
# Load with HuggingFace datasets
ds = load_dataset("csv", data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")["train"]

# Add combined_text column: prompt + space + A
ds = ds.map(
    lambda row: {
        "combined_text": row["prompt"] + " " + row["A"]
    }
)

# Character length at index 51
len(ds[51]["combined_text"])

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

614

# Q2. BERT tokenizer vocabulary size

In [4]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

tokenizer.vocab_size

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

30522

# Q3. Integer ID of the [SEP] token

In [5]:
tokenizer.sep_token_id

102

# Q4. Shape of input_ids tensor when tokenizing entire prompt column

In [7]:
prompts = list(ds["prompt"])

encoded = tokenizer(
    prompts,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

encoded["input_ids"].shape

torch.Size([2000, 128])

# Q5. Dimensionality of each attention head in bert-base-uncased

Hidden size = 768, number of heads = 12 -> each head = 768 / 12 = 64

# Q6. Shape of last_hidden_state for the prompt at row index 0

In [8]:
bert_model = AutoModel.from_pretrained("bert-base-uncased")
bert_model.eval()

row0_prompt = ds[0]["prompt"]

# Tokenize with default settings — no manual padding or truncation
inputs_row0 = tokenizer(row0_prompt, return_tensors="pt")

with torch.no_grad():
    outputs_row0 = bert_model(**inputs_row0)

outputs_row0.last_hidden_state.shape

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


torch.Size([1, 31, 768])

# Q7. Sum of first 5 float values in the [CLS] embedding

In [9]:
cls_vector = outputs_row0.last_hidden_state[0, 0, :]  # batch 0, token 0 ([CLS])

cls_vector[:5].tolist()

[-0.4676641821861267,
 -0.07544498145580292,
 -0.20190085470676422,
 -0.007064265664666891,
 -0.4480222165584564]

In [10]:
round(cls_vector[:5].sum().item(), 4)

-1.2001

# Q8. Attention weight [CLS] → 'fusion' in the last layer, head 0

Sentence: "Light-ion fusion is a technique."

In [11]:
bert_attn = AutoModel.from_pretrained("bert-base-uncased", output_attentions=True)
bert_attn.eval()

test_sentence = "Light-ion fusion is a technique."
inputs_test = tokenizer(test_sentence, return_tensors="pt")

# Check which token index corresponds to 'fusion'
tokens = tokenizer.convert_ids_to_tokens(inputs_test["input_ids"][0])
print(f"Tokens: {tokens}")
fusion_idx = tokens.index("fusion")
print(f"'fusion' is at token index: {fusion_idx}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Tokens: ['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']
'fusion' is at token index: 4


In [12]:
with torch.no_grad():
    outputs_test = bert_attn(**inputs_test)

# Last layer = index -1, first head = index 0
attn_last_layer = outputs_test.attentions[-1]  # shape: (batch, heads, seq, seq)
attn_head0 = attn_last_layer[0, 0]             # shape: (seq, seq)

# Attention from [CLS] (index 0) to fusion token
round(attn_head0[0, fusion_idx].item(), 4)

0.1025

# Q9. Cosine similarity between prompt and Option B at row 0 using SentenceTransformer

In [13]:
st_model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

row0_b = ds[0]["B"]

emb_prompt = st_model.encode(row0_prompt, convert_to_tensor=True)
emb_b      = st_model.encode(row0_b,      convert_to_tensor=True)

round(st_util.cos_sim(emb_prompt, emb_b).item(), 4)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

0.7658

# Q11. Zero-shot classification - single-label (Softmax)

Use the prompt from row index 1 with options A, B, C as candidate labels.

In [14]:
zs_pipeline = pipeline("zero-shot-classification")  # defaults to facebook/bart-large-mnli

row1_prompt = ds[1]["prompt"]
row1_A = ds[1]["A"]
row1_B = ds[1]["B"]
row1_C = ds[1]["C"]
candidate_labels = [row1_A, row1_B, row1_C]

print(f"Prompt: {row1_prompt}")
print(f"Candidates: {candidate_labels}")

result_softmax = zs_pipeline(row1_prompt, candidate_labels)

round(result_softmax["scores"][0], 4)

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Prompt: What is accelerator-based light-ion fusion?
Candidates: ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion

0.4575

# Q12. Zero-shot classification — multi-label (Sigmoid)

Same call but with multi_label=True. Find |sum_softmax - sum_sigmoid|.

In [15]:
result_sigmoid = zs_pipeline(row1_prompt, candidate_labels, multi_label=True)
print("Sigmoid result:")
print(result_sigmoid)

sum_softmax = sum(result_softmax["scores"])
sum_sigmoid = sum(result_sigmoid["scores"])

round(abs(sum_softmax - sum_sigmoid), 4)

Sigmoid result:
{'sequence': 'What is accelerator-based light-ion fusion?', 'labels': ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to i

0.9995

# Q13. Generative AI — flan-t5-small text2text generation

Construct the exact prompt string for row index 0 and pass to flan-t5-small.

In [18]:
from transformers.pipelines import SUPPORTED_TASKS
SUPPORTED_TASKS.keys()

dict_keys(['audio-classification', 'automatic-speech-recognition', 'text-to-audio', 'feature-extraction', 'text-classification', 'token-classification', 'question-answering', 'table-question-answering', 'visual-question-answering', 'document-question-answering', 'fill-mask', 'text-generation', 'zero-shot-classification', 'zero-shot-image-classification', 'zero-shot-audio-classification', 'image-classification', 'image-feature-extraction', 'image-segmentation', 'image-text-to-text', 'object-detection', 'zero-shot-object-detection', 'depth-estimation', 'video-classification', 'mask-generation', 'image-to-image', 'keypoint-matching', 'any-to-any'])

In [ ]:
# text2text-generation is not available